# PPO Atari

Notebook with a demonstration PPO implementation, running on several gymnasium environments

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium.wrappers import DtypeObservation, RescaleObservation, ResizeObservation, GrayscaleObservation, FrameStackObservation, AtariPreprocessing
import matplotlib.pyplot as plt
import os
import sys
import time
from typing import List, Callable, Dict, Optional
from gymnasium.spaces.utils import flatten, flatdim
import ale_py

from IPython.display import clear_output
from live_view import LiveView

from importlib import reload
sys.path.append(os.getcwd())
import PPO

gym.register_envs(ale_py)

In [ ]:
plt.style.use('dark_background')

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(torch.cuda.is_available())

reload(PPO)

In [ ]:
ENV_NAME = "ale_py:PongNoFrameskip-v4"
#ENV_NAME = "ale_py:BreakoutNoFrameskip-v4"

ENV_SETUPS = {
    "ale_py:PongNoFrameskip-v4" : 
        {
            "noop_max":30,
            "frame_skip":4,
            "screen_size":84,
            "terminal_on_life_loss":False,
            "grayscale_obs":True,
            "scale_obs":True,
        },
    "ale_py:BreakoutNoFrameskip-v4" :
        {
            "noop_max":30,
            "frame_skip":4,
            "screen_size":84,
            "terminal_on_life_loss":True,
            "grayscale_obs":True,
            "scale_obs":True,
        }
}

In [ ]:
# Playable cell of the environment
from neptune_nb.gym import play

# env = gym.make(ENV_NAME, render_mode="rgb_array")
# returns = play(env, {"space": 1, "down": 3, "up": 2}, noop=0, episodes=1, fps="60", scale=1.0)   # 2 = main engine
# env.close()
# print("return:", round(returns[0], 1))

In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            PPO.layer_init(nn.Conv2d(4, 32, kernel_size=8, stride=4)),
            nn.ReLU(),
            PPO.layer_init(nn.Conv2d(32, 64, kernel_size=4, stride=2)),
            nn.ReLU(),
            PPO.layer_init(nn.Conv2d(64, 64, kernel_size=3, stride=1)),
            nn.ReLU(),
            nn.Flatten(),
            PPO.layer_init(nn.Linear(64 * 7 * 7, 512)),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.cnn(x)

class VisionActor(nn.Module):
    def __init__(self, obs_dim: int, n_actions: int, cnn: CNNEncoder, hidden: int = 64):
        #super().__init__(obs_dim, n_actions)
        super().__init__()
        self.cnn = cnn
        print(f"Obs dim: {obs_dim}, Actions Dim: {n_actions}")
        self.net = nn.Sequential(
            PPO.layer_init(nn.Linear(512, hidden)), nn.Tanh(),
            PPO.layer_init(nn.Linear(hidden, n_actions), std=0.01),
        )

    def forward(self, x):
        features = self.cnn(x)
        logits = self.net(features)  # logits
        return logits
    def dist(self, x):
        return Categorical(logits=self.forward(x))

class VisionCritic(nn.Module):
    def __init__(self, obs_dim: int, cnn: CNNEncoder, hidden: int = 64):
        super().__init__()
        #super().__init__(obs_dim)
        self.cnn = cnn
        self.net = nn.Sequential(
            PPO.layer_init(nn.Linear(512, hidden)), nn.Tanh(),
            PPO.layer_init(nn.Linear(hidden, 1), std=1.0),
        )

    def forward(self, x):
        features = self.cnn(x)
        return self.net(features).squeeze(-1)

def make_env_callback(env: gym.Env):
    if ENV_NAME in ENV_SETUPS:
        print("Using environment setup for ", ENV_NAME)
        c = ENV_SETUPS[ENV_NAME]
        env = AtariPreprocessing(
            env,
            noop_max=c["noop_max"],
            frame_skip=c["frame_skip"],
            screen_size=c["screen_size"],
            terminal_on_life_loss=c["terminal_on_life_loss"],
            grayscale_obs=c["grayscale_obs"],
            scale_obs=c["scale_obs"],
        )
    else:
        env = AtariPreprocessing(
            env,
            noop_max=30,
            frame_skip=4,
            screen_size=84,
            terminal_on_life_loss=True,
            grayscale_obs=True,
            scale_obs=True,
        )
    env = FrameStackObservation(env, 4)
    return env

In [ ]:
# Test to make sure our observations will work with the network
#test_env = gym.make(ENV_NAME)
# test_env = GrayscaleObservation(test_env)
# test_env = ResizeObservation(test_env, (84, 84))
# test_env = DtypeObservation(test_env, dtype=np.float32)
# test_env = RescaleObservation(test_env, min_obs=0.0, max_obs=1.0)
# test_env = AtariPreprocessing(
    # test_env,
    # noop_max=30,
    # frame_skip=4,
    # screen_size=84,
    # terminal_on_life_loss=True,
    # grayscale_obs=True,
    # scale_obs=True,
# )
def make_one(i):
    def _thunk():
        env = gym.make(ENV_NAME)
        env.action_space.seed(0)
        env = make_env_callback(env)
        return env
    return _thunk
test_envs = gym.vector.SyncVectorEnv([make_one(i) for i in range(2)])

obs, _ = test_envs.reset()
print(obs.shape)

obs_dim = test_envs.observation_space.shape
print(obs_dim)
# should have (envs, x, y)
print(test_envs.single_observation_space.shape)
n_actions = test_envs.single_action_space.n
cnn = CNNEncoder().to(device) # Shared cnn encoder
actor = VisionActor(obs_dim, n_actions, cnn=cnn).to(device)
critic = VisionCritic(obs_dim, cnn=cnn).to(device)

obs_t = torch.from_numpy(obs).to(device)
#obs_t = obs_t.unsqueeze(1) # Add channel dimension
print(obs_t.shape)
# #trainer.actor(obs)
test = cnn(obs_t)
print("CNN output shape: ", test.shape)
# #print(test)
test2 = actor(obs_t)
print("Actor output shape: ", test2.shape)

In [ ]:
ppoTrainerConfig = PPO.PPOTrainerConfig()
ppoTrainerConfig.num_envs = 8
ppoTrainerConfig.async_envs = True
ppoTrainerConfig.make_env_callback = make_env_callback
ppoTrainerConfig.device = device
trainer = PPO.PPOTrainer(ENV_NAME, ppoTrainerConfig)

# Create the actor and critic architectures
obs_dim = trainer.env.single_observation_space.shape
n_actions = trainer.env.single_action_space.n
print(f"Obs Dimension: {obs_dim}, Actions Dimensions: {n_actions}")
cnn = CNNEncoder().to(device) # Shared cnn encoder
trainer.actor = VisionActor(obs_dim, n_actions, cnn=cnn).to(device)
trainer.critic = VisionCritic(obs_dim, cnn=cnn).to(device)

# Set common Atari training parameters
trainConfig = PPO.PPOTrainerTrainConfig()
trainConfig.total_steps = 3_000_000
trainConfig.rollout_steps = 128 # per environment
trainConfig.lr = 2.5e-4
trainConfig.adam_eps = 1e-5
ppoConfig = PPO.PPOTrainerPPOUpdateConfig()
ppoConfig.minibatch_size = 256
ppoConfig.epochs = 4
ppoConfig.clip_eps = 0.1
ppoConfig.ent_coef = 0.01
ppoConfig.vf_coef = 0.5
ppoConfig.max_grad_norm = 0.5

print("Start training...")
start_time = time.time()
run = trainer.train(trainConfig, ppoConfig)['returns']
elapsed_time = time.time() - start_time
print("Training complete. Elapsed time: ", elapsed_time)

In [ ]:
def plot_returns(runs: Dict[str, List[float]], smooth: int = 20, title: str = ""):
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, returns in runs.items():
        if len(returns) < smooth:
            ax.plot(returns, alpha=0.6, label=label)
            continue
        r = np.asarray(returns)
        smoothed = np.convolve(r, np.ones(smooth) / smooth, mode='valid')
        ax.plot(np.arange(smooth - 1, len(r)), smoothed, label=label)
    ax.set_xlabel("Episode")
    ax.set_ylabel(f"Return (smoothed over {smooth})")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_returns({"Run" : run})

In [ ]:
def make_env(name: str, render: bool) -> gym.Env:
    render_mode = "rgb_array" if render else None
    env = gym.make(name, render_mode=render_mode)
    return env

In [ ]:
# Execute running our a greedy policy based on Q

def convert_observation(obs, env: gym.Env) -> torch.tensor:
    """Converts an integer observation to a tensor"""
    return torch.as_tensor(obs).to(device)

def evaluate(env: gym.Env, actor: PPO.Actor, render = True) -> float:
    actor.eval()
    done = False
    G_t = 0

    with torch.no_grad():
        S, info = env.reset()
        S = convert_observation(S, env)

        view = LiveView(env) if render else None
        if render:
            view.show(env.render())

        while not done:
            S = S.unsqueeze(0)
            action_dist = actor.dist(S)
            A = action_dist.sample().item()
            S_p, reward, terminated, trunc, info = env.step(A)
            G_t += reward
            S_p = convert_observation(S_p, env)

            S = S_p

            if render:
                view.show(env.render())

            if terminated or trunc:
                done = True
    
    if render:
        view.close()

    # return the total return
    return G_t

In [ ]:
#env = PPO.make_envs(ENV_NAME, 1, False, make_env_callback=make_env_callback, render_mode="rgb_array")
env = make_env(ENV_NAME, True)
env = make_env_callback(env)

In [ ]:
G = evaluate(env, trainer.actor, render=True)
print("Total Return: ", G)